# Experiment #2: Comparison LLMS

In this experiment, we compare the performance of different Taxonomies created by LLMs on the task of generating recommendations and the similarity of the generated Taxonomies. We going to use as a recommendation Model the KGIL model from the first experiment. For a neat evaluation, we will use the amazon-video_games dataset. It has more items than the other datasets, so the LLMs have more information to generate the Taxonomies. We assume that this will help the LLMs to a wider range of variations of the Taxonomies and be more challenging for the approach.

**Data**: We use the amazon-video_games dataset. Since LLMs are computationally expensive, we use a smaller subset of the dataset for the LLMs. We use a 20-core subset of the dataset.

**Models**: We use the following LLMs:
+ Llama3.2-3b: _In experiment #1 we presented our approach to generate Taxonomies from text using Keyword Extraction Model and LLMs. As baseline LLM we used the open weightLlama3.2-3b Model. It is the small version of the LLama3.2 family with a footprint of roughly 2 GB which makes it the smallest LLMs in the experiment
+ ChatGPT-4: _This model was used in different studies, showed very promising results in the generation of Taxonomies [1](https://arxiv.org/pdf/2402.07386), [2](https://arxiv.org/pdf/2403.12173). It was able to outperform classic approaches and other state-of-the-art LLMs. [[3]](https://arxiv.org/pdf/2503.21810)_ 
+ DeepSeek-R1: _This open weight showed in the comparsion stduy [[3]](https://arxiv.org/pdf/2503.21810) of LLMs integrated in Embedding Models to generate taxonomies on tabular data promising results. On one dataset itwas able to generate a taxonomy the highest RI and Purity. It has a footprint of 4.7 GB_ 
+ Qwen2.5-7B: _Additionally we explore the effectiveness of the Qwen2.5-Modell with 7B parameters. In the comparsion study [[3]](https://arxiv.org/pdf/2503.21810) the model family with 14B and 32B parameters achieved good results, while trailing to the GPT-4 and DeepSeek-R1, for taxnomies on tabular data. It has a footprint of 4.7 GB_ 

**Metrics**: We use the following metrics:
- Quantitative: 
+ Performance and cost: We use the same performance and cost metrics as in the first experiment.
+ Similarity: As qualitative metric we use the `levenshtein distance` and `cosine similarity` to measure the similarity of the generated Taxonomies.

- Qualitative:
+ Error analysis: We conduct an error analysis on the KGIL model for each generated Taxonomy. This will help identify if the taxonomies are useful during the multi-hop reasoning of the KGIL model. Also it help identify if KGIL systematically predicts errors for certain entities.

All the metrics help identify the best performing LLM for the task of generating Taxonomies and how to improve the performance of the LLM taxonomie genration process for Graph Neural Network Knowledge Graph Recommender Systems.

**Infrastructure:**
The experiments were conducted on a Apple M2 MacBook Pro with 16GB RAM and 10-core CPU.

## .0 Setup

In [3]:
import os # path management
import sys # system path management
import pandas as pd # data manipulation
import numpy as np # data manipulation
from scipy.sparse import csr_matrix # build adjency matrix
import random # random number generation
import torch # for gnn
import networkx as nx # graph library
from sentence_transformers import SentenceTransformer # 

from time import time # time measurement
import Levenshtein # levenshtein distance


sys.path.append('../..')  # Add the project root to the path so we can import from utils

from utils.create_triplets_from_hierarchies import PrepareData


In [1]:
# --- Configuration ---
# Set the desired data path, dataset name and model technique here
DATASET = "amazon-Video_Games"

MODEL_TECHNIQUES = ["llama3.2", "chatgpt-4", "deepseek-r1", "qwen2.5"]
MODEL_TECHNIQUE = MODEL_TECHNIQUES[1].replace('/', '_')
current_dir = os.getcwd()
project_root = os.path.dirname(os.path.dirname(os.path.dirname(current_dir)))

EXPERIMENT_NAME = "#2"
EXPERIMENT_DATA_PATH = os.path.join(project_root, 'data', 'experiments',EXPERIMENT_NAME,DATASET,MODEL_TECHNIQUE)

NameError: name 'os' is not defined

# playground

In [7]:
current_dir = os.getcwd()
project_root = os.path.dirname(os.path.dirname(os.path.dirname(current_dir)))

data_path = os.path.join(project_root, 'data', 'preprocessed', 'amazon-All_Beauty')

metadata_filtered_df = pd.read_csv(data_path+'/metadata_filtered_df.csv')


In [8]:
LLM_MODEL = "gemma3"

In [27]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import OllamaLLM
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer
import pandas as pd
import os
import time
import numpy as np


class LLMHierarchyCOL:
    def __init__(self, llm_name: str):
        # initialize LLM once
        self.llm = OllamaLLM(model=llm_name, temperature=0.1)

        # --- detailed summary chain ---
        detailed_system = """
You are an expert that generates detailed product descriptions. 
You receive a short compact product description. The detailed product description will be later used to extract Named Entities or Key Words to build a taxonomy

Side information: the product description includes ingredients (e.g almond oil), usage information (e. g. silky hair), package dimensions etc
Be creative. Make it about a paragraph long

Rules:
- Don't include explanations or notes
- Don't hallucinate 
- Only add valuable information to the product
- Prioritize sizing or packaging dimensions less, but still include them
"""
        detailed_user = "This is the description of the product. {product_description}"
        detail_tpl = ChatPromptTemplate(
            messages=[("system", detailed_system), ("human", detailed_user)],
            input_variables=["product_description"]
        )
        self.detail_chain = detail_tpl | self.llm

        # --- generate taxonomy chain ---
        gen_system = """
You are a helpful assistant and expert in constructing a taxonomy from a given concept.
You will build iteratively a taxonomy with a root concept and from the taxonomy in the previous step and the entity list.

The format of the generated taxonomy is: 
1. Parent Concept 
1.1 Child Concept.
1.1.1 Grandchild Concept.
1.1.2 Grandchild Concept.
1.2 Child Concept.
1.2.1 Grandchild Concept.
1.2.2 Grandchild Concept.
Do not change any entity names when building the taxonomy.

Critical rules 
- DO NOT ADD ANY COMMENTS OR EXPLANATIONS, ONLY RETURN THE TAXONOMY
- THERE IS ONE AND ONLY ONE ROOT NODE OF THE TAXONOMY 
- ALL ENTITIES FROM THE LIST MUST APPEAR IN THE TAXONOMY IF FEASIBLE
- EXCLUDE ENTITIES WITH NO INFORMATION DETAIL FOR THE PARENT CONCEPT
- YOU ARE ALLOWED TO ADD OTHER ENTITIES IF THEY MAKE SENSE, DON'T HALLUCINATE
- ONLY RETURN THE TAXONOMY
"""
        gen_user = """
The root entity is {root_concept}, the taxonomy in the current step is:
{taxonomy}

The entity list for this step contains:
{list_entities}
"""
        gen_tpl = ChatPromptTemplate(
            messages=[("system", gen_system), ("user", gen_user)],
            input_variables=["root_concept", "taxonomy", "list_entities"]
        )
        self.gen_chain = gen_tpl | self.llm

        # --- update taxonomy chain ---
        update_system = """
You are a helpful assistant and expert in updating a given hierarchical taxonomy with a root concept.
Review the given taxonomy from the previous step.

If the taxonomy contains unreasonable or wrong relations, update them to make sense.
You may relocate child nodes to new parents and add nodes if necessary.
The format of the generated taxonomy is: 
1. Parent Concept 
1.1 Child Concept.
1.1.1 Grandchild Concept.
1.1.2 Grandchild Concept.
1.2 Child Concept.
1.2.1 Grandchild Concept.
1.2.2 Grandchild Concept.
Do not change any entity names when building the taxonomy.

Critical rules 
- DO NOT ADD ANY COMMENTS OR EXPLANATIONS 
- THERE IS ONE AND ONLY ONE ROOT NODE 
- ALL ENTITIES MUST APPEAR IF FEASIBLE
- EXCLUDE ENTITIES WITH NO DETAIL
- KEEP THE NUMBERING FORMAT
- ONLY RETURN THE TAXONOMY
"""
        update_user = """
The root entity is {root_concept}.
The entity list contains: {list_entities}

Current taxonomy:
{taxonomy}
"""
        update_tpl = ChatPromptTemplate(
            messages=[("system", update_system), ("user", update_user)],
            input_variables=["root_concept", "list_entities", "taxonomy"]
        )
        self.update_chain = update_tpl | self.llm

        # --- review taxonomy chain ---
        review_system = """
You are an expert for validating a generated hierarchical taxonomy.
Your task is to check if the taxonomy suits the product description.

If it's unsuitable, return FALSE.
If it's suitable, return TRUE.

Critical rules: 
- Do not include any explanations or notes 
- ONLY RETURN THE BOOLEAN 
- Do not hallucinate 
- Validate carefully

Output Format:
BOOLEAN
"""
        review_user = """
The root entity is {root_concept}.
Entity list: {list_entities}

Taxonomy to review:
{taxonomy}

Product description:
{product_description}
"""
        review_tpl = ChatPromptTemplate(
            messages=[("system", review_system), ("user", review_user)],
            input_variables=["root_concept", "list_entities", "taxonomy", "product_description"]
        )
        self.review_chain = review_tpl | self.llm

        # --- embedding & keyword models ---
        self.embed_model = SentenceTransformer('all-MiniLM-L6-v2')
        self.embed_kw_model = SentenceTransformer("sentence-transformers/paraphrase-mpnet-base-v2")
        self.kw_model = KeyBERT(model=self.embed_kw_model)

    def generate_details(self, short_summary: str) -> str:
        return self.detail_chain.invoke({"product_description": short_summary})

    def get_keywords(self, detailed_summary: str) -> list:
        raw = self.kw_model.extract_keywords(
            detailed_summary,
            keyphrase_ngram_range=(1, 2),
            top_n=5,
            stop_words="english",
            use_maxsum=True
        )
        # return unique keywords
        return list({kw for kw, _ in raw})

    def generate_tax(self, root_concept: str, list_keywords: list, taxonomy: str) -> str:
        return self.gen_chain.invoke({
            "root_concept": root_concept,
            "taxonomy": taxonomy,
            "list_entities": list_keywords
        })

    def update_tax(self, taxonomy: str, root_concept: str, list_keywords: list) -> str:
        return self.update_chain.invoke({
            "root_concept": root_concept,
            "list_entities": list_keywords,
            "taxonomy": taxonomy
        })

    def review(self, detailed_summary: str, taxonomy: str, root_concept: str, list_keywords: list) -> bool:
        out = self.review_chain.invoke({
            "root_concept": root_concept,
            "list_entities": list_keywords,
            "taxonomy": taxonomy,
            "product_description": detailed_summary
        })
        return out.strip().upper() == "TRUE"

    def taxonomy_to_triples(self, taxonomy_text: str) -> pd.DataFrame:
        lines = taxonomy_text.strip().split("\n")
        triples = []
        hierarchy = {}
        for line in lines:
            if not line.strip():
                continue
            parts = line.strip().split(" ", 1)
            if len(parts) < 2:
                continue
            level_num, concept = parts
            level_num = level_num.rstrip(".")
            hierarchy[level_num] = concept
            if "." in level_num:
                parent_level = ".".join(level_num.split(".")[:-1])
                if parent_level in hierarchy:
                    triples.append({
                        "head": hierarchy[parent_level],
                        "relation": "is parent of",
                        "tail": concept
                    })
        return pd.DataFrame(triples, columns=["head", "relation", "tail"])

    def linkage_asin_to_taxonomy(
        self,
        triples_df: pd.DataFrame,
        dict_asin_keywords: dict,
        sim_thresh: float = 0.5
    ) -> pd.DataFrame:
        heads = triples_df["head"].tolist()
        tails = triples_df["tail"].tolist()
        H = self.embed_model.encode(heads)  # (n, d)
        T = self.embed_model.encode(tails)  # (n, d)

        new_links = []
        for asin, kws in dict_asin_keywords.items():
            K = self.embed_model.encode(kws)  # (k, d)
            sim_h = K @ H.T                  # (k, n)
            sim_t = K @ T.T                  # (k, n)
            # find all keyword→taxonomy matches over threshold
            idxs_h = np.argwhere(sim_h > sim_thresh)
            idxs_t = np.argwhere(sim_t > sim_thresh)
            for i, j in idxs_h:
                new_links.append({"head": asin, "relation": "related to", "tail": heads[j]})
            for i, j in idxs_t:
                new_links.append({"head": asin, "relation": "related to", "tail": tails[j]})

        if new_links:
            out = pd.concat([triples_df, pd.DataFrame(new_links)], ignore_index=True)
            return out.drop_duplicates().reset_index(drop=True)
        return triples_df


if __name__ == "__main__":
    # Hyperparameters
    DATASETS = ["Books", "All_Beauty", "Video_Games", "Last-FM"]
    DATASET = DATASETS[1]  # e.g. "All_Beauty"
    DIR_NAME = "amazon-"
    LLM_MODEL = "gemma3"
    SENTENCE_TRANSFORMER = "all-MiniLM-L6-v2"
    ROOT_CONCEPT = "Product details"

    # Paths
    current_dir = os.getcwd()
    data_path = os.path.join(
        current_dir,
        "data",
        "preprocessed",
        f"{DIR_NAME}{DATASET}",
        "metadata_filtered_df.csv"
    )
    #metadata_filtered_df = pd.read_csv(data_path)

    llm_hierarchy = LLMHierarchyCOL(LLM_MODEL)

    start = time.time()

    # 1) Generate detailed summaries (streaming to save memory)
    for idx, row in metadata_filtered_df.iloc[:10].iterrows():
        print(f"Detailing row {idx}")
        metadata_filtered_df.at[idx, "detailed_summary"] = (
            llm_hierarchy.generate_details(row.source_text)
        )
        # optionally flush to disk every N rows...

    # 2) Batch‐wise taxonomy building
    BATCH_SIZE = 5
    dict_asin_keywords = {}
    taxonomy_text = ""

    for i in range(0, len(metadata_filtered_df.iloc[:10]), BATCH_SIZE):
        print(f"Processing batch {i} to {i + BATCH_SIZE}")
        batch = metadata_filtered_df.iloc[i : i + BATCH_SIZE]
        batch_kw_lists = []
        for _, item in batch.iterrows():
            kws = llm_hierarchy.get_keywords(item.detailed_summary)
            batch_kw_lists.append(kws)
            dict_asin_keywords[item.parent_asin] = kws

        # generate → update → review
        taxonomy_text = llm_hierarchy.generate_tax(ROOT_CONCEPT, batch_kw_lists, taxonomy_text)
        print(taxonomy_text)
        taxonomy_text = llm_hierarchy.update_tax(taxonomy_text, ROOT_CONCEPT, batch_kw_lists)
        ok = llm_hierarchy.review(batch.iloc[0].detailed_summary, taxonomy_text, ROOT_CONCEPT, batch_kw_lists)
        if not ok:
            taxonomy_text = llm_hierarchy.generate_tax(ROOT_CONCEPT, batch_kw_lists, "")
            ok = llm_hierarchy.review(batch.iloc[0].detailed_summary, taxonomy_text, ROOT_CONCEPT, batch_kw_lists)

        # transform → link → save
        triples_df = llm_hierarchy.taxonomy_to_triples(taxonomy_text)
        linked_df = llm_hierarchy.linkage_asin_to_taxonomy(triples_df, dict_asin_keywords)

        out_path = os.path.join(
            current_dir,
            "data",
            "relations",
            f"{DIR_NAME}{DATASET}",
            f"{LLM_MODEL}_taxonomy_triples_batch{i}.csv"
        )
        # makedirs
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        linked_df.to_csv(out_path, index=False)

    end = time.time()
    print(f"Total runtime: {end - start:.1f}s")
    print(f"Per‐item runtime: {(end - start) / len(metadata_filtered_df):.3f}s")


Detailing row 0
Detailing row 1
Detailing row 2
Detailing row 3
Detailing row 4
Detailing row 5
Detailing row 6
Detailing row 7
Detailing row 8
Detailing row 9
Processing batch 0 to 5
1. Product details
1.1 Hair care
1.1.1 deeply moisturizes
1.1.2 almond fractionated
1.1.3 coconut
1.2 oils cleaning
1.2.1 oils cleaning
1.2.2 glass
1.2.3 oz container
1.3 combs featuring
1.3.1 transform shower
1.3.2 leaving hair
1.3.3 static detangling
1.3.4 grooming experience
1.4 night cream
1.4.1 tula probiotic
1.4.2 revitalized skin
1.4.3 radiance complexion
1.4.4 intense hydration
1.5 style clip
1.5.1 style clip
1.5.2 slippery strands
1.5.3 ikoco pack
1.5.4 fine hair
1.5.5 secure jaw
Processing batch 5 to 10
1. Product details
1.1 Hair care
1.1.1 deeply moisturizes
1.1.2 almond fractionated
1.1.3 coconut
1.1.4 oil bundle
1.2 oils cleaning
1.2.1 oils cleaning
1.2.2 oz container
1.2.3 glass
1.2.4 hand wash
1.3 combs featuring
1.3.1 transform shower
1.3.2 leaving hair
1.3.3 static detangling
1.3.4 groom

In [29]:
triples_df_gemma = triples_df.copy()

In [17]:
triples_df_qwen = triples_df.copy()

In [32]:
def levenshtein_distance(df1, df2):

    def collect_unique_values(df):
        # collect from parent_name, child_left_name, child_right_name all unique values
        unique_values = []
        for column in ['head', 'tail']:
            unique_values.extend(df[column].unique())
            # remove Parent Asin values like this B07KG1TWP5
            unique_values = [value for value in unique_values if not value.startswith('B')]
            # only unique values
            unique_values = list(set(unique_values))
        return unique_values
    unique_values_df1 = collect_unique_values(df1)
    unique_values_df2 = collect_unique_values(df2)
    print(unique_values_df1)
    print(unique_values_df2)
    print('-'*20)
    lev_ratio =Levenshtein.ratio(unique_values_df1, unique_values_df2)

    return lev_ratio

print('qwen2.5_gemma3_lev_ratio')
qwen2_gemma3_lev_ratio = levenshtein_distance(triples_df_qwen, triples_df_gemma)
print(qwen2_gemma3_lev_ratio)

qwen2.5_gemma3_lev_ratio
['Revitalizing scalp', 'Lightweight breathable', 'Care package', 'Maintain hairstyle', 'Pippa london', 'Powder convenience', 'Elevate lash', 'Magic star', 'Multivitamins almond', 'Massage', 'Use pencil', 'Curler features', 'Product details', 'Skincare', 'Showers shower', 'Comfortable cleanse', 'Eyelash care', 'Skin types', 'Jdo eyelash', 'Mascara value', 'Cap women', 'Facial care', 'Disposable plastic', 'Grooming', 'Teqifu hair', 'Makeup enthusiasts', 'Hair care']
['Makeup', 'experience shower', 'eyeshadow powder', 'silk fiber', 'ounces shower', 'jdo eyelash', 'Shower Accessories', 'brush set', 'lasts pencil', 'polyester stylish', 'Eye Care', 'hair care', 'magic star', 'Product details', 'Facial Kits', 'radiance pitta', 'skin golden', 'teqifu pack', 'kit luxurious', 'gift box', 'mascara value', 'revitalizing scalp', 'curler features', 'reusable', 'nourishing facial', 'shower luxurious', 'Hair care', 'cap women', 'pencil used', 'makeup wipes', '4d silk']
-------

In [30]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Fix the cosine_similarity function to handle matrix inputs correctly
def cosine_similarity_matrices(matrix1, matrix2):
    # Transpose the second matrix to align dimensions for dot product
    matrix2_transposed = matrix2.T
    # Calculate dot product between matrices
    dot_product = np.dot(matrix1, matrix2_transposed)
    
    # Calculate norms for each row in both matrices
    norm_matrix1 = np.linalg.norm(matrix1, axis=1, keepdims=True)
    norm_matrix2 = np.linalg.norm(matrix2, axis=1, keepdims=True)
    
    # Calculate cosine similarity
    cosine_sim = dot_product / (np.dot(norm_matrix1, norm_matrix2.T))
    
    return cosine_sim

# Update the calc_cosine_similarity function to use the correct similarity function
def calc_cosine_similarity_fixed(df1, df2, embedding_model):
    def collect_unique_values(df):
        # collect from parent_name, child_left_name, child_right_name all unique values
        unique_values = []
        for column in ['head', 'tail']:
            unique_values.extend(df[column].unique())
            # remove Parent Asin values like this B07KG1TWP5
            unique_values = [value for value in unique_values if not value.startswith('B')]
        return unique_values
    
    unique_values_df1 = collect_unique_values(df1)
    unique_values_df2 = collect_unique_values(df2)

    # embed unique values
    unique_values_df1_embed = embedding_model.encode(unique_values_df1)
    unique_values_df2_embed = embedding_model.encode(unique_values_df2)

    # Use the fixed cosine similarity function for matrices
    cosine_similarity_matrix = cosine_similarity_matrices(unique_values_df1_embed, unique_values_df2_embed).mean()

    return cosine_similarity_matrix

# Use the fixed function
print(f"Cosine similarity between qwen2.5 and gemma3: {calc_cosine_similarity_fixed(triples_df_qwen, triples_df_gemma, embedding_model)}")


Cosine similarity between qwen2.5 and gemma3: 0.24058346450328827


## 1. Experiment

### 1.1 DeepSeek-R1

In [ ]:

data_path = os.path.join(project_root, 'data', 'preprocessed', f'{DATASET}')
metadata_df = pd.read_csv(data_path+'/metadata_filtered_df.csv')
train_df = pd.read_csv(data_path+'/train_df.csv')
test_df = pd.read_csv(data_path+'/test_df.csv')

# Load relations
triplet_path = os.path.join(project_root, 'data', 'relations', f'{DATASET}')
all_mini_triplet_df = pd.read_csv(triplet_path+'/'+f'{MODEL_TECHNIQUE}_relations.csv')

prepare_data = PrepareData()
train_df = prepare_data.user_item_triples(train_df)
test_df = prepare_data.user_item_triples(test_df)
kg_df, train_adj, test_adj, name_to_id_map = prepare_data.build_kg(all_mini_triplet_df, train_df, test_df,EXPERIMENT_DATA_PATH)


In [ ]:

sys.path.append('../../src/models/KGIL')
from prettytable import PrettyTable
from models.KGIL.utils.parser import parse_args
from models.KGIL.utils.data_loader import load_data
from models.KGIL.utils.evaluate_kgil import test
from models.KGIL.utils.helper import early_stopping
from models.KGIL.model import EnvGenerator, Recommender

# --- Dynamic Argument Setting ---
# Store the original sys.argv
original_argv = sys.argv.copy()

# Clear existing arguments (except the script name itself) and add the desired ones
# sys.argv[0] is typically the path to the ipykernel launcher, keep it.
sys.argv = [sys.argv[0]]
sys.argv.extend(['--data_path', EXPERIMENT_DATA_PATH])
sys.argv.extend(['--dataset', DATASET])
sys.argv.extend(['--model_technique', MODEL_TECHNIQUE])
n_users = 0
n_items = 0
n_entities = 0
n_nodes = 0
n_relations = 0


def get_feed_dict(train_entity_pairs, start, end, train_user_set):

    def negative_sampling(user_item, train_user_set):
        neg_items = []
        for user, _ in user_item.cpu().numpy():
            user = int(user)
            while True:
                neg_item = np.random.randint(low=0, high=n_items, size=1)[0]
                if neg_item not in train_user_set[user]:
                    break
            neg_items.append(neg_item)
        return neg_items

    feed_dict = {}
    entity_pairs = train_entity_pairs[start:end].to(device)
    feed_dict['users'] = entity_pairs[:, 0]
    feed_dict['pos_items'] = entity_pairs[:, 1]
    feed_dict['neg_items'] = torch.LongTensor(negative_sampling(entity_pairs,
                                                                train_user_set)).to(device)
    return feed_dict


def build_graph(train_cf, kg_dict):
    print("\nDetailed graph building process:")
    
    graph = nx.MultiDiGraph()
    print(f"Initial graph state: {len(graph.nodes())} nodes, {len(graph.edges())} edges")
    
    # Add user-item interactions
    for u, i in train_cf:
        if not graph.has_node(u):
            graph.add_node(u, type='user')
        if not graph.has_node(i):
            graph.add_node(i, type='item')
        graph.add_edge(u, i, type='interact')
    
    print(f"After adding interactions: {len(graph.nodes())} nodes, {len(graph.edges())} edges")
    
    # Add knowledge graph triples
    for h, r, t in kg_dict:
        if not graph.has_node(h):
            graph.add_node(h, type='entity')
        if not graph.has_node(t):
            graph.add_node(t, type='entity')
        graph.add_edge(h, t, type=r)
    
    print(f"After adding KG: {len(graph.nodes())} nodes, {len(graph.edges())} edges")
    
    # Verify node types
    node_types = {}
    for node in graph.nodes():
        node_type = graph.nodes[node].get('type', 'unknown')
        node_types[node_type] = node_types.get(node_type, 0) + 1
    print("\nNode type distribution:", node_types)
    
    return graph


def train_one_epoch(model, train_cf, user_dict, args):
    model.train()
    
    # Adjust batch size
    batch_size = min(args.batch_size, len(train_cf))
    n_batch = len(train_cf) // batch_size + 1
    
    total_loss = 0
    for batch_idx in range(n_batch):
        start = batch_idx * batch_size
        end = min((batch_idx + 1) * batch_size, len(train_cf))
        
        if start >= end:
            break
            
        batch = get_feed_dict(train_cf, start, end, user_dict['train_user_set'])
        
        # Print batch statistics
        print(f"\nBatch {batch_idx + 1}/{n_batch}:")
        print(f"Users: {batch['users'].shape}, range: [{batch['users'].min()}, {batch['users'].max()}]")
        print(f"Pos items: {batch['pos_items'].shape}, range: [{batch['pos_items'].min()}, {batch['pos_items'].max()}]")
        print(f"Neg items: {batch['neg_items'].shape}, range: [{batch['neg_items'].min()}, {batch['neg_items'].max()}]")
        
        rec_loss, inv_mean, inv_var = model(batch)
        outer_loss = rec_loss + args.lamda * (inv_mean + inv_var)
        
        # Print loss components
        print(f"Loss components:")
        print(f"Rec loss: {rec_loss.item():.6f}")
        print(f"Inv mean: {inv_mean.item():.6f}")
        print(f"Inv var: {inv_var.item():.6f}")
        print(f"Total loss: {outer_loss.item():.6f}")
        
        if outer_loss.requires_grad:
            model.zero_grad()
            outer_loss.backward()
            
            # Print gradient norms
            total_norm = 0
            for p in model.parameters():
                if p.grad is not None:
                    param_norm = p.grad.data.norm(2)
                    total_norm += param_norm.item() ** 2
            total_norm = total_norm ** 0.5
            print(f"Gradient norm: {total_norm:.6f}")
            
            model.optimizer.step()
            
        total_loss += outer_loss.item()
    
    return total_loss / n_batch


if __name__ == '__main__':
    """fix the random seed"""
    seed = 2020
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    """read args"""
    global args, device
    args = parse_args()
    device = torch.device("cuda:" + str(args.gpu_id) if torch.cuda.is_available() and args.cuda else "cpu")

    """build dataset"""
    train_cf, test_cf, user_dict, n_params, graph, mat_list = load_data(args)
    adj_mat_list, norm_mat_list, mean_mat_list = mat_list

    print(f"Dataset statistics:")
    print(f"Number of training interactions: {len(train_cf)}")
    print(f"Number of test interactions: {len(test_cf)}")
    print(f"Number of users: {n_params['n_users']}")
    print(f"Number of items: {n_params['n_items']}")
    print(f"Batch size: {args.batch_size}")

    n_users = n_params['n_users']
    n_items = n_params['n_items']
    n_entities = n_params['n_entities']
    n_relations = n_params['n_relations']
    n_nodes = n_params['n_nodes']
    
    """cf data"""
    train_cf_pairs = torch.LongTensor(np.array([[cf[0], cf[1]] for cf in train_cf], np.int32))
    test_cf_pairs = torch.LongTensor(np.array([[cf[0], cf[1]] for cf in test_cf], np.int32))

    """define model"""
    augmenter = EnvGenerator(args.K, args.dim, args.dim, device)
    model = Recommender(n_params, args, graph, mean_mat_list[0], device).to(device)
    
    """define optimizer"""
    rec_optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    aug_optimizer = torch.optim.Adam(augmenter.parameters(), lr=args.lr)
    model.augmenter = augmenter

    cur_best_pre_0 = 0
    stopping_step = 0
    should_stop = False
    total_iter = max(len(train_cf) // args.batch_size, 1)
    get_print = max(total_iter // 20, 1)
    
    print("start training ...")
    for epoch in range(args.epoch):
        # Add epoch start print
        print(f"\nEpoch {epoch+1}/{args.epoch}")
        print(f"Number of iterations expected: {total_iter}")
        
        index = np.arange(len(train_cf))
        np.random.shuffle(index)
        model.train()

        train_cf_pairs = train_cf_pairs[index]
        total_outer_loss, total_inner_loss, s = 0, 0, 0
        train_s_t = time()
        iteration = 0
        
        # Add batch processing print
        print(f"Processing {len(train_cf)} samples in batches of {args.batch_size}")
        
        while s + args.batch_size <= len(train_cf):
            batch = get_feed_dict(train_cf_pairs, s, s + args.batch_size, user_dict['train_user_set'])
            
            # Debug print for first batch of first epoch
            if epoch == 0 and iteration == 0:
                print("\nFirst batch statistics:")
                print(f"Users shape: {batch['users'].shape}")
                print(f"Positive items shape: {batch['pos_items'].shape}")
                print(f"Negative items shape: {batch['neg_items'].shape}")
            
            rec_loss, inv_mean, inv_var = model(batch)
            outer_loss = rec_loss + args.lamda * (inv_mean + inv_var)
            
            # Debug print losses
            if iteration % 100 == 0:
                print(f"\nIteration {iteration} losses:")
                print(f"rec_loss: {rec_loss.item():.6f}")
                print(f"inv_mean: {inv_mean.item():.6f}")
                print(f"inv_var: {inv_var.item():.6f}")
                print(f"outer_loss: {outer_loss.item():.6f}")
            
            rec_optimizer.zero_grad()
            outer_loss.backward()
            
            # Check gradients periodically
            if iteration % 10 == 0:
                print("\nGradient norms:")
                for name, param in model.named_parameters():
                    if param.grad is not None:
                        grad_norm = param.grad.data.norm(2).item()
                        print(f"{name}: {grad_norm:.6f}")
            
            rec_optimizer.step()
            total_outer_loss += outer_loss.item()

            if epoch % args.epup == 0 and iteration > 10 and iteration % args.itup == 0:
                # Debug augmenter update
                rec_loss, inv_mean, inv_var = model(batch)
                inner_loss = - inv_var
                
                if iteration == 12:  # First augmenter update
                    print(f"\nAugmenter update:")
                    print(f"inner_loss: {inner_loss.item():.6f}")
                    print(f"inv_var: {inv_var.item():.6f}")
                
                aug_optimizer.zero_grad()
                inner_loss.backward()
                
                # Debug augmenter gradients
                if iteration == 12:
                    print("\nAugmenter gradient norms:")
                    for name, param in augmenter.named_parameters():
                        if param.grad is not None:
                            print(f"{name}: {param.grad.data.norm(2).item():.6f}")
                
                aug_optimizer.step()
                total_inner_loss += inner_loss.item()

            s += args.batch_size
            iteration += 1
            inner_loss = - inv_var
            if iteration % get_print == 0:
                print("Train epoch:[{}/{}], iter:[{}/{}], outer_loss:[{:.6f}], inner_loss:[{:.6f}], time:[{:.2f}] min"
                .format(epoch + 1, args.epoch, 
                        iteration, total_iter, 
                        outer_loss.item(), 
                        inner_loss.item(),
                        (time() - train_s_t) / 60))

        train_e_t = time()
        print("-" * 100)
        print("start evluation ...")
        print("-" * 100)
        test_s_t = time()
        ret = test(model, user_dict, n_params)
        test_e_t = time()
        epoch_outer_loss = total_outer_loss  / total_iter
        epoch_inner_loss = total_inner_loss  / total_iter
        train_time = (train_e_t - train_s_t) / 60
        test_time = (test_e_t - test_s_t) / 60
        
        # Create a pretty table for metrics
        metrics_table = PrettyTable()
        metrics_table.field_names = ["Metric", "Value"]
        
        # Add all available metrics
        for k_idx, k in enumerate(eval(args.Ks)):
            metrics_table.add_row([f"Recall@{k}", f"{ret['recall'][k_idx]:.4f}"])
            metrics_table.add_row([f"NDCG@{k}", f"{ret['ndcg'][k_idx]:.4f}"])
            metrics_table.add_row([f"Precision@{k}", f"{ret['precision'][k_idx]:.4f}"])
            metrics_table.add_row([f"Hit_Ratio@{k}", f"{ret['hit_ratio'][k_idx]:.4f}"])
        
        metrics_table.add_row(["AUC", f"{ret['auc']:.4f}"])
        metrics_table.add_row(["Train Time (min)", f"{train_time:.2f}"])
        metrics_table.add_row(["Test Time (min)", f"{test_time:.2f}"])
        metrics_table.add_row(["Outer Loss", f"{epoch_outer_loss:.6f}"])
        metrics_table.add_row(["Inner Loss", f"{epoch_inner_loss:.6f}"])
        
        print("\nEvaluation Metrics:")
        print(metrics_table)
        print("-" * 100)
        
        # Keep the original print for consistency
        recall_20 = ret['recall'][0]
        ndcg_20 = ret['ndcg'][0]
        print("Test epoch:[{}/{}], loss:[out:{:.6f}, in:{:.6f}], recall:[{:.4f}], ndcg:[{:.4f}] | train time:[{:.2f}] min, test time:[{:.2f}] min"
            .format(epoch + 1, args.epoch, 
                    epoch_outer_loss, epoch_inner_loss,
                    recall_20, ndcg_20,
                    train_time, test_time))
        print("-" * 100)
        

## 1.2 ChatGPT-4

## 1.3 Qwen2.5-7B

## 2. Results

## 3. Interpretation